## 3.05 Boolean Expressions: completed work

**Adhvay Iyer · RazorCrest00 · September 29, 2026**

[Assignment](https://pages.opencodingsociety.com/python/boolean/2027) · [Download executed notebook](https://raw.githubusercontent.com/RazorCrest00/portfolio/main/_notebooks/homework/2026-09-29-boolean-hw.ipynb)

**Self-assessment: 1.00 / 1.00.** The evidence below maps completed work to the assignment; this is not a teacher-issued grade. Examples and explanations were prepared with AI assistance. Popcorn solutions are recorded here without claiming unverified classroom participation.

Each Python cell runs independently. Saved output is included in the notebook; **Run** produces fresh output in this page. The examples use classroom data, not live backend records.


## Popcorn 1: Check every required field

**Prediction:** The original part has a name, allowed category, spec number, and date, so all four checks are `True`. Changing only the category to `Street Car` makes only the category check `False`. Whitespace-only text is treated as empty.


In [1]:
# CODE_RUNNER: 3.05 Popcorn 1 - Field checks
def field_checks(part):
    return {
        "product name": bool(part.get("product_name", "").strip()),
        "racing category": part.get("category") in ["Auto Racing", "Drag Racing"],
        "spec number": bool(part.get("spec_number", "").strip()),
        "effective date": bool(part.get("effective_date", "").strip()),
    }

part = {"product_name": "Replacement Flywheels and Clutch Assemblies",
        "category": "Auto Racing", "spec_number": "1.1", "effective_date": "Nov. 9, 2001"}
for label, candidate in [("Original", part), ("Changed category", dict(part, category="Street Car"))]:
    print(label)
    for field, valid in field_checks(candidate).items():
        print(field + ":", valid)


Original
product name: True
racing category: True
spec number: True
effective date: True
Changed category
product name: True
racing category: False
spec number: True
effective date: True


## Popcorn 2: Combine the create rule

**Prediction:** Spec `1.1` is rejected because it already exists; changing it to `1.2` allows creation when all fields remain valid. `and` requires every check; `not` reverses the duplicate flag. The validator checks a snapshot and does not write to a database.


In [2]:
# CODE_RUNNER: 3.05 Popcorn 2 - Create or reject
def validate_record(part, existing_spec_numbers, valid_categories):
    has_name = bool(part.get("product_name", "").strip())
    has_spec = bool(part.get("spec_number", "").strip())
    has_date = bool(part.get("effective_date", "").strip())
    category_ok = part.get("category") in valid_categories
    duplicate_spec = part.get("spec_number", "").strip() in existing_spec_numbers
    return has_name and has_spec and has_date and category_ok and not duplicate_spec

existing_spec_numbers = ["1.1", "2.1"]
valid_categories = ["Auto Racing", "Drag Racing"]
part = {"product_name": "Replacement Flywheels and Clutch Assemblies",
        "category": "Auto Racing", "spec_number": "1.1", "effective_date": "Nov. 9, 2001"}
print("Existing spec:", validate_record(part, existing_spec_numbers, valid_categories))
print("New spec:", validate_record(dict(part, spec_number="1.2"), existing_spec_numbers, valid_categories))


Existing spec: False
New spec: True


## Popcorn 3: Search with grouped Boolean logic

**Prediction:** Query `1.2` finds the clutch record. Query `flywheel` finds two records; an unknown query finds none. A matching name in an invalid category must still be excluded. The rule is `(name_match or exact_spec_match) and category_ok`, so category filtering applies to both kinds of match. Name matching ignores case; spec matching is exact.


In [3]:
# CODE_RUNNER: 3.05 Popcorn 3 - Search filter
def search_records(records, query, valid_categories):
    query = query.strip()
    matches = []
    for record in records:
        name_match = bool(query) and query.casefold() in record["product_name"].casefold()
        spec_match = bool(query) and query == record["spec_number"]
        category_ok = record["category"] in valid_categories
        if (name_match or spec_match) and category_ok:
            matches.append(record)
    return matches

records = [
    {"product_name": "Replacement Flywheels and Clutch Assemblies", "category": "Auto Racing", "spec_number": "1.1"},
    {"product_name": "Multiple Disc Clutch Assemblies", "category": "Drag Racing", "spec_number": "1.2"},
    {"product_name": "Racing Flywheel Record", "category": "Auto Racing", "spec_number": "2.1"},
]
valid_categories = ["Auto Racing", "Drag Racing"]
for query in ["1.2", "flywheel", "missing", ""]:
    matches = search_records(records, query, valid_categories)
    print(repr(query) + ":", [record["spec_number"] for record in matches])
invalid = dict(records[0], category="Street Car")
print("Invalid category excluded:", search_records([invalid], "flywheel", valid_categories) == [])


'1.2': ['1.2']
'flywheel': ['1.1', '2.1']
'missing': []
'': []
Invalid category excluded: True


## Homework: Reusable SFI validator

**Prediction:** The three supplied candidates produce **accepted, rejected, rejected**: a new valid spec passes, a street-car category fails, and an existing spec fails. The function returns an actual Boolean; presentation is separate from validation. Inputs are dictionaries with string-valued fields. A date must be present; this exercise does not require parsing or authenticating the date.


In [4]:
# CODE_RUNNER: 3.05 Homework - SFI backend validator
def validate_record(part, existing_spec_numbers, valid_categories):
    has_name = bool(part.get("product_name", "").strip())
    has_spec = bool(part.get("spec_number", "").strip())
    has_date = bool(part.get("effective_date", "").strip())
    category_ok = part.get("category") in valid_categories
    duplicate_spec = part.get("spec_number", "").strip() in existing_spec_numbers
    return has_name and has_spec and has_date and category_ok and not duplicate_spec

existing_spec_numbers = ["1.1", "2.1"]
valid_categories = ["Auto Racing", "Drag Racing"]
test_records = [
    {"product_name": "Multiple Disc Clutch Assemblies", "category": "Auto Racing", "spec_number": "1.2", "effective_date": "Feb. 9, 2006"},
    {"product_name": "Replacement Flywheels", "category": "Street Car", "spec_number": "3.1", "effective_date": "Jan. 1, 2026"},
    {"product_name": "Existing Flywheel Record", "category": "Drag Racing", "spec_number": "1.1", "effective_date": "Jan. 1, 2026"},
]
for candidate in test_records:
    valid = validate_record(candidate, existing_spec_numbers, valid_categories)
    if valid:
        print(candidate["spec_number"] + ": ACCEPTED")
    else:
        print(candidate["spec_number"] + ": REJECTED")


1.2: ACCEPTED
3.1: REJECTED
1.1: REJECTED


## Tests and AP connection

A valid record becomes invalid when any required field is missing or blank, the category is rejected, or the spec is duplicated. The checks below exercise each reason separately. They also test an allowed Drag Racing record and a whitespace-padded duplicate. AP uses `AND`, `OR`, `NOT`, and `=` for Boolean equality; Python uses `and`, `or`, `not`, and `==`.


In [5]:
# CODE_RUNNER: 3.05 Tests - Missing fields and duplicates
def validate_record(part, existing_spec_numbers, valid_categories):
    has_name = bool(part.get("product_name", "").strip())
    has_spec = bool(part.get("spec_number", "").strip())
    has_date = bool(part.get("effective_date", "").strip())
    category_ok = part.get("category") in valid_categories
    duplicate_spec = part.get("spec_number", "").strip() in existing_spec_numbers
    return has_name and has_spec and has_date and category_ok and not duplicate_spec

base = {"product_name": "Practice clutch", "category": "Auto Racing", "spec_number": "9.1", "effective_date": "Sep. 29, 2026"}
existing = ["1.1", "2.1"]
allowed = ["Auto Racing", "Drag Racing"]
cases = [("valid", base, True), ("other allowed category", dict(base, category="Drag Racing"), True),
         ("duplicate", dict(base, spec_number="1.1"), False),
         ("padded duplicate", dict(base, spec_number=" 1.1 "), False),
         ("invalid category", dict(base, category="Street Car"), False)]
for field in ["product_name", "spec_number", "effective_date"]:
    cases.append(("blank " + field, dict(base, **{field: "   "}), False))
    missing = base.copy()
    del missing[field]
    cases.append(("missing " + field, missing, False))
for name, record, expected in cases:
    result = validate_record(record, existing, allowed)
    assert type(result) is bool and result is expected, name
    print(name + ": PASS (" + str(result) + ")")


valid: PASS (True)
other allowed category: PASS (True)
duplicate: PASS (False)
padded duplicate: PASS (False)
invalid category: PASS (False)
blank product_name: PASS (False)
missing product_name: PASS (False)
blank spec_number: PASS (False)
missing spec_number: PASS (False)
blank effective_date: PASS (False)
missing effective_date: PASS (False)


## MCQ: 4 / 4

`MCQ 3.05: 4/4 | answers: A,C,A,B`

| Question | Answer and reason |
| --- | --- |
| `7 >= 7` | A: `True`; equality is included. |
| Require name and spec | C: `and` requires both. |
| `not False` | A: `True`; negation reverses it. |
| Either racing category | B: `or` allows either comparison. |

The answers were checked in the lesson's client-side quiz with AI assistance; this is not a teacher-issued grade.


## Full-credit justification and extension evidence

**Why this supports full credit:** The three Popcorn tasks establish the individual field checks, the complete creation rule, and grouped search logic. The homework combines all required fields, accepted categories, and duplicate rejection in one reusable Boolean procedure. Its supplied cases yield accepted, rejected, rejected; the recorded MCQ is 4/4. Those results support each row of the existing 1.00-point rubric.

**Above and beyond the required result:** The extra checks isolate missing and whitespace-only fields, an allowed second category, and a padded duplicate spec. Separating the Boolean result from printed labels makes the rule reusable and testable. Search groups the name/spec alternatives before category filtering, preventing an invalid category from passing through one side of `or`.

### Reviewer challenge: change the input

Using `validate_record`, start with a record named `Practice Clutch`, category `Auto Racing`, spec `9.1`, and a nonempty date. With existing specs `["1.1"]`, expect **True**. Change only the date to spaces: **False**. Restore the date and add `9.1` to existing specs: **False**. Each rejection has a different reason.

**Verification:** This challenge is checked against the actual submitted solution by [the submission-evidence verifier](https://github.com/RazorCrest00/portfolio/blob/main/scripts/test_submission_justifications.py). Reset the runner afterward to restore the assignment sample.

**Advanced delivery:** The existing OCS/Pyodide integration lets a reviewer edit Python and obtain fresh results inside the portfolio. Saved outputs support notebook review; browser execution supports independent reproduction. This is integration and verification work, with the editor/runtime credited to OCS/Pyodide.

**Scope of the claim:** The rule validates the supplied string fields and a snapshot of existing specs. It requires a date but does not authenticate it or write to a database. The evidence supports a self-assessment; the final awarded grade remains the grader’s decision.

**Submission-ready justification:** All Popcorn tasks, the reusable validator, and the 4/4 MCQ are complete; isolated invalid-field and duplicate tests demonstrate the combined rule beyond the sample.


## Rubric evidence

| Criterion | Self-score | Evidence |
| --- | ---: | --- |
| Popcorn 1 | 0.10 / 0.10 | Four visible field checks; changed category |
| Popcorn 2 | 0.15 / 0.15 | Valid create and duplicate rejection |
| Popcorn 3 | 0.20 / 0.20 | Grouped query/category rule; changed and empty queries |
| MCQ | 0.15 / 0.15 | 4/4 result and reasoning |
| Homework validator | 0.40 / 0.40 | Reusable Boolean validator, supplied cases, and 11 checks |


## Design thinking and browser execution

**Need:** Make the decision or calculation clear enough to inspect. **Goal:** Correct results for ordinary and boundary inputs. **Choice:** Keep each task self-contained and give important decisions descriptive names. **Prototype:** The completed cells above. **Test and revision:** Compare expected outputs with execution, then check changed inputs and boundary cases; task-specific findings are recorded above.

The page uses the portfolio's existing OCS editor and Pyodide runtime to execute Python in the browser. This makes the submitted work editable and reproducible beyond static code. OCS/Pyodide provide that infrastructure; no new CSS or shared runner changes are claimed. An initial runtime download is required. These are code checks, not a claim of a peer playtest.


## Submission notes

**Page:** https://razorcrest00.github.io/portfolio/python/boolean-hw

[Download the self-contained homework code](https://razorcrest00.github.io/portfolio/assets/homework/2026-09-29/boolean.py). Where the [lesson](https://pages.opencodingsociety.com/python/boolean/2027) has a Code Submission form, choose Python and paste the file contents without Markdown fences. Include this page link and the following notes. The form has not been submitted automatically.

```text
Lesson: 3.05 Boolean Expressions
Portfolio: https://razorcrest00.github.io/portfolio/python/boolean-hw
MCQ 3.05: 4/4 | answers: A,C,A,B
Homework supplied cases: ACCEPTED, REJECTED, REJECTED
Popcorn work: completed and tested in this notebook
Tests: see saved outputs and verification evidence
Self-assessment: see criterion-by-criterion table
```

**Sources:** [Current assignment](https://pages.opencodingsociety.com/python/boolean/2027), checked September 29, 2026; [Python language reference](https://docs.python.org/3/reference/); [College Board pseudocode reference](https://apcentral.collegeboard.org/media/pdf/ap-computer-science-principles-exam-reference-sheet.pdf). Attribution includes the assignment's supplied data and starter structure.

<script>
document.addEventListener('DOMContentLoaded', () => {
  document.querySelectorAll('.code-runner-container .languageSelect').forEach(select => {
    select.replaceChildren(new Option('Python', 'python'));
  });
});
</script>
